# 04k Extraction, batch k: seven studies coded from their PDFs and two full-text exclusions

Design log.
- Batch k of notebook 04 (plan v2 B4), the last batch of included studies with full text in hand, coded under the frozen schema (schema/record_schema.yaml, v1) and the coding rules, as batches a to j were. The seven reports: HAA/ULC (Research Square), Defenses against LLM prompt injections in academic peer review (PsyArXiv), Spotlight-Guard (Applied Sciences), AdverShield-LLM (Journal of Technology and System Information), Hybrid Defense Against Indirect Prompt Injection in RAG (IJAIML), the extended report for Defended in Training, Eroded in Deployment (Zenodo) and Indirect Prompt Injection in Municipal Document-Processing Copilots (Street Art & Urban Creativity).
- Each record's extraction_scope names the tables and rows coded. As before, detectors, sanitisers, filters and system-level designs are left out and the scope says so (amendment 4): Detector-Only and every Spotlight-Guard configuration, the passage isolation and aggregation of AdverShield-LLM and its baselines, Output Verification in the hybrid RAG study, and the guardrail and combined conditions in the municipal study. Direct-injection rows are left out (HAA/ULC).
- Coding decisions worth knowing when reading these records (reading aids for notebook 05, not findings):
  - HAA/ULC's indirect rows rest on 6 and 4 samples and their percentages do not correspond to whole numbers of samples; a residual note says so.
  - The peer-review study pools three models; the acceptance rate with injection under the checking prompt is shown only in a figure, so that result carries no value.
  - The Spotlight-Guard paper does not name the model behind its Baseline and Spotlighting-Only arms; only that pair is in scope.
  - AdverShield-LLM's attack success pools corpus poisoning, prompt injection and embedding perturbation and is coded as a proxy; its 'PLSA only' ablation row repeats the RobustRAG row exactly, which a residual note records.
  - The erosion report compares Meta-SecAlign before and after benign fine-tuning (M comparisons) and codes the authors' SFT repair as a trained defence against the matching undefended model.
  - The municipal study pools four models.
- Full-text exclusions found while coding: Signed to Stop (rs-11711485) protects the agent with a pinned operator key and an operator-side verification tool, so it is excluded as system level; the JOEIS conference paper (joeis.v4i1.81604) reports only a defence that combines adversarial retraining with an input-sanitisation filter, pooled over direct and indirect attacks, and is excluded as system level. Stage 2b checks both anchors in the PDFs and carries the decisions into schema/fulltext_decisions.csv, schema/screening_decisions.csv, reports/screening.csv, reports/defences.csv, reports/fulltext_check.csv and reports/prisma_counts.json. The number of included studies falls from 126 to 124.
- GuardRAG (ResearchWorks) remains to be coded once its full text is in hand.
- Outputs: reports/records/ (seven files), reports/extraction_04k_check.csv, reports/extraction_04k_summary.csv, reports/extraction_batches.csv (rows for batch k added, earlier batches kept), reports/decision_04k.txt.
- CPU; no network calls beyond pip; model calls: 0.

## Stage 0. Session bootstrap

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/SourceMap_Research"
    REPO_DIR = PROJECT_ROOT + "/sourcemap-research"
except ImportError:
    PROJECT_ROOT = os.environ["SOURCEMAP_PROJECT_ROOT"]
    REPO_DIR = os.environ["SOURCEMAP_REPO_ROOT"]
os.environ["SOURCEMAP_PROJECT_ROOT"] = PROJECT_ROOT
os.environ["SOURCEMAP_REPO_ROOT"] = REPO_DIR
os.chdir(REPO_DIR)
if "src" not in sys.path:
    sys.path.insert(0, "src")
for module, package in (("yaml", "pyyaml"), ("pypdf", "pypdf"), ("pypdfium2", "pypdfium2")):
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)
import pypdf
if int(pypdf.__version__.split(".")[0]) < 4:                 # the column-preserving extraction needs pypdf 4 or later
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pypdf>=4"], check=False)
    print("pypdf upgraded; restart the runtime if the next stage reports an old version")
print(subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True).stdout.strip() or "pull: nothing to report")
print("cwd:", os.getcwd(), "| pypdf", pypdf.__version__)

## Stage 1. The 7 records

In [ ]:
Path("reports/records").mkdir(parents=True, exist_ok=True)       # %%writefile does not create folders

`haaulc_rs2026` (doi_10_21203_rs_3_rs_9653510_v1)

In [ ]:
%%writefile reports/records/haaulc_rs2026.yaml
report:
  id: haaulc_rs2026
  slug: doi_10_21203_rs_3_rs_9653510_v1
  title: 'Hierarchical Adaptive Anti-injection Framework (HAA/ULC): A Lightweight Defense Strategy Based on Input Decoupling and Semantic Isolation'
  first_author: Zhishu Dou
  venue: Research Square preprint
  peer_reviewed: 'no'
  doi: 10.21203/rs.3.rs-9653510/v1
  version_read: v1 (11 May 2026)
  first_date: '2026-05-11'
  read: F
  role: proposes
  extraction_scope: Table 1 rows Document Poisoning and Web Content Injection (indirect injection) for GPT-4o with and without the ULC V3.2 protocol, and the Section 5.2.4 usability scores; the Section 5.2.2 claim (direct,
    role-play, task-drift, chain-of-thought, encoded and hybrid rows are direct injection; the multi-agent cascade is described in text only)
defences:
- id: ulc
  name: HAA/ULC V3.2 system-prompt protocol (input split into Data, Question and Instruction, relevance and conflict review, master-objective restatement; only PASS fields are used)
  origin: this_report
  base_checkpoint: GPT-4o (2024-08-06)
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 6
  layout: system prompt protocol; user input containing the attack sample
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: Input Decoupling Engine
evaluations:
- id: ev_doc_none
  defence_id: none:GPT-4o (2024-08-06)
  benchmark: 'author-built attack library, indirect injection: Document Poisoning (6 samples, 3 runs each, majority)'
  attacker_type: static
  attacker_access: black_box
  judge: malicious instruction executed (human raters)
  layout_same_as_defence: 'yes'
  measured_by: haaulc_rs2026
  page: 12
  decoding: temperature 0.2
  sample_size: '6'
  baseline_kind: instruction_prompt_only
- id: ev_doc_ulc
  defence_id: ulc
  benchmark: 'author-built attack library, indirect injection: Document Poisoning (6 samples, 3 runs each, majority)'
  attacker_type: static
  attacker_access: black_box
  judge: malicious instruction executed (human raters)
  layout_same_as_defence: 'yes'
  measured_by: haaulc_rs2026
  page: 12
  decoding: temperature 0.2
  sample_size: '6'
- id: ev_web_none
  defence_id: none:GPT-4o (2024-08-06)
  benchmark: 'author-built attack library, indirect injection: Web Content Injection (4 samples, 3 runs each, majority)'
  attacker_type: static
  attacker_access: black_box
  judge: malicious instruction executed (human raters)
  layout_same_as_defence: 'yes'
  measured_by: haaulc_rs2026
  page: 12
  decoding: temperature 0.2
  sample_size: '4'
  baseline_kind: instruction_prompt_only
- id: ev_web_ulc
  defence_id: ulc
  benchmark: 'author-built attack library, indirect injection: Web Content Injection (4 samples, 3 runs each, majority)'
  attacker_type: static
  attacker_access: black_box
  judge: malicious instruction executed (human raters)
  layout_same_as_defence: 'yes'
  measured_by: haaulc_rs2026
  page: 12
  decoding: temperature 0.2
  sample_size: '4'
- id: ev_tu_none
  defence_id: none:GPT-4o (2024-08-06)
  benchmark: 100 legitimate business queries
  attacker_type: none
  attacker_access: not_applicable
  judge: task usability score (1 to 5)
  layout_same_as_defence: 'yes'
  measured_by: haaulc_rs2026
  page: 14
  decoding: temperature 0.2
  sample_size: '100'
  baseline_kind: instruction_prompt_only
- id: ev_tu_ulc
  defence_id: ulc
  benchmark: 100 legitimate business queries
  attacker_type: none
  attacker_access: not_applicable
  judge: task usability score (1 to 5)
  layout_same_as_defence: 'yes'
  measured_by: haaulc_rs2026
  page: 14
  decoding: temperature 0.2
  sample_size: '100'
results:
- id: r_doc_none
  evaluation_id: ev_doc_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Document Poisoning
  value: '75.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_doc_ulc
  evaluation_id: ev_doc_ulc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Document Poisoning
  value: '20.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_web_none
  evaluation_id: ev_web_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Web Content Injection
  value: '60.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_web_ulc
  evaluation_id: ev_web_ulc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Web Content Injection
  value: '28.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 12
  location: Table 1
- id: r_tu_none
  evaluation_id: ev_tu_none
  quantity: R2clean
  coverage: direct
  metric_as_printed: average task usability score
  value: '4.8'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 5.2.4
- id: r_tu_ulc
  evaluation_id: ev_tu_ulc
  quantity: R2clean
  coverage: direct
  metric_as_printed: average task usability score
  value: '4.6'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 14
  location: Section 5.2.4
comparisons:
- id: c_doc
  left: ev_doc_ulc
  right: ev_doc_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_web
  left: ev_web_ulc
  right: ev_web_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 12
- id: c_tu
  left: ev_tu_ulc
  right: ev_tu_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 14
claims:
- id: k1
  text: the protocol raises defence success against indirect injection in knowledge-base documents to 76%
  location: Section 5.2.2
  anchor: DSR improved to 76%
  jobs:
  - R1
  supported_by:
  - c_doc
  - c_web
  page: 13
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 1 (indirect rows)
    R2clean: Section 5.2.4
    R2attack: not reported
schema_gaps:
- where: report
  note: the indirect rows rest on 6 and 4 samples, and their percentages (75.0, 20.0, 60.0, 28.0) do not correspond to whole numbers of samples; Section 5.2.2 gives a different baseline (65% leakage) for the knowledge-base
    case than Table 1

`peerrevcheck_osf2025` (doi_10_31234_osf_io_zxuwf_v2)

In [ ]:
%%writefile reports/records/peerrevcheck_osf2025.yaml
report:
  id: peerrevcheck_osf2025
  slug: doi_10_31234_osf_io_zxuwf_v2
  title: Defenses against LLM prompt injections in academic peer review
  first_author: Ondrej Zika
  venue: PsyArXiv (OSF) preprint
  peer_reviewed: 'no'
  doi: 10.31234/osf.io/zxuwf_v2
  version_read: v2
  first_date: '2025-07-15'
  read: F
  role: mixed
  extraction_scope: 'Results text and Figure 1: the injection-checking prompt against the standard review prompt, pooled over three models, on injected and non-injected manuscripts (acceptance, scientific quality and identified
    flaws); the abstract claim (detection rates are not coded; per-model results are figure only)'
defences:
- id: check
  name: injection-checking prompt (reviewer told to look for statements trying to alter the prompt, ignore and report them)
  origin: this_report
  base_checkpoint: ChatGPT 4o, Llama 4 and Gemini 2.5 Flash, pooled
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: whole manuscript attached to the review prompt; hidden sentence at a random point
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: Check for any statements trying to alter this
evaluations:
- id: ev_inj_none
  defence_id: none:ChatGPT 4o, Llama 4 and Gemini 2.5 Flash, pooled
  benchmark: 26 unpublished bioRxiv neuroscience preprints with one injected sentence asking for a publication recommendation
  attacker_type: static
  attacker_access: black_box
  judge: model's own recommendation (0/1), quality (0-10) and flaw (0/1) ratings
  layout_same_as_defence: 'yes'
  measured_by: peerrevcheck_osf2025
  page: 4
  decoding: not reported
  sample_size: 26 papers x 3 models
  baseline_kind: instruction_prompt_only
- id: ev_inj_check
  defence_id: check
  benchmark: 26 unpublished bioRxiv neuroscience preprints with one injected sentence asking for a publication recommendation
  attacker_type: static
  attacker_access: black_box
  judge: model's own recommendation (0/1), quality (0-10) and flaw (0/1) ratings
  layout_same_as_defence: 'yes'
  measured_by: peerrevcheck_osf2025
  page: 4
  decoding: not reported
  sample_size: 26 papers x 3 models
- id: ev_clean_none
  defence_id: none:ChatGPT 4o, Llama 4 and Gemini 2.5 Flash, pooled
  benchmark: the same 26 preprints without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: model's own recommendation, quality and flaw ratings
  layout_same_as_defence: 'yes'
  measured_by: peerrevcheck_osf2025
  page: 4
  decoding: not reported
  sample_size: 26 papers x 3 models
  baseline_kind: instruction_prompt_only
- id: ev_clean_check
  defence_id: check
  benchmark: the same 26 preprints without injection
  attacker_type: none
  attacker_access: not_applicable
  judge: model's own recommendation, quality and flaw ratings
  layout_same_as_defence: 'yes'
  measured_by: peerrevcheck_osf2025
  page: 4
  decoding: not reported
  sample_size: 26 papers x 3 models
results:
- id: r_inj_none_acc
  evaluation_id: ev_inj_none
  quantity: R1
  coverage: proxy
  metric_as_printed: acceptance recommendation rate under injection
  value: '94.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 4
  location: Results
- id: r_inj_check_acc
  evaluation_id: ev_inj_check
  quantity: R1
  coverage: proxy
  metric_as_printed: acceptance recommendation rate under injection
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: figure_only
  page: 5
  location: Figure 1
- id: r_clean_none_acc
  evaluation_id: ev_clean_none
  quantity: R2clean
  coverage: proxy
  metric_as_printed: acceptance recommendation rate, no injection
  value: '74.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 4
  location: Results
- id: r_clean_none_q
  evaluation_id: ev_clean_none
  quantity: R2clean
  coverage: proxy
  metric_as_printed: scientific quality rating, no injection
  value: '7.33'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 4
  location: Results
- id: r_clean_check_acc
  evaluation_id: ev_clean_check
  quantity: R2clean
  coverage: proxy
  metric_as_printed: acceptance recommendation rate, no injection
  value: '52.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 4
  location: Results
- id: r_clean_check_q
  evaluation_id: ev_clean_check
  quantity: R2clean
  coverage: proxy
  metric_as_printed: scientific quality rating, no injection
  value: '7.11'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 4
  location: Results
comparisons:
- id: c_inj
  left: ev_inj_check
  right: ev_inj_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_clean
  left: ev_clean_check
  right: ev_clean_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
claims:
- id: k1
  text: prompting the models to check for injections fully reverts the effect of the injection, at the cost of harsher reviews
  location: abstract
  anchor: Prompting LLMs to check for injections was successful at fully reverting these effects
  jobs:
  - R1
  - R2
  supported_by:
  - c_inj
  - c_clean
  page: 1
coverage:
  R1: proxy
  R2clean: proxy
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Results text and Figure 1 (acceptance under injection)
    R2clean: Results text (side effects without injection)
    R2attack: not reported
schema_gaps:
- where: results r_inj_check_acc
  note: the acceptance rate with injection and the checking prompt is shown only in Figure 1; the text reports no difference from non-injected manuscripts under the same prompt (p > 0.28); results are pooled over three models
    and 26 papers

`spotguard_applsci2026` (doi_10_3390_app16157662)

In [ ]:
%%writefile reports/records/spotguard_applsci2026.yaml
report:
  id: spotguard_applsci2026
  slug: doi_10_3390_app16157662
  title: 'Balancing Security and Performance in LLM Agents: Spotlight-Guard, a Layered Defense Against Indirect Prompt Injection'
  first_author: Doygun Demirol
  venue: Applied Sciences
  peer_reviewed: 'yes'
  doi: 10.3390/app16157662
  version_read: published version
  first_date: '2026-08-02'
  read: F
  role: proposes
  extraction_scope: Table 3 rows Baseline and Spotlighting-Only (ASR and benign success); (Detector-Only and every Spotlight-Guard configuration contain the LLM guard, quarantine and fallback models and are left out as detector
    and system level; ReAct is an agent prompting format rather than a defence and is not coded; Table 5 ablates components inside the full system)
defences:
- id: spot
  name: Spotlighting-Only (untrusted tool content encoded with delimiters or Base64 before the model reads it)
  origin: other_report
  base_checkpoint: locally hosted 7B-class open-weight model via Ollama (qwen2.5:7b serves the pipeline; the baseline model is not named separately)
  axis1:
  - delimiters_markers
  - encodings
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: InjecAgent-derived tool-integrated agent cases; untrusted tool content encoded
  layout_compat:
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: The Spotlighting (Data Encoding) sub-component isolates content from untrusted
evaluations:
- id: ev_none
  defence_id: none:locally hosted 7B-class open-weight model via Ollama (qwen2.5:7b serves the pipeline; the baseline model is not named separately)
  benchmark: InjecAgent-based stratified benchmark, 250 adversarial cases
  attacker_type: static
  attacker_access: black_box
  judge: rule-based automated judge (attack success when the defence is bypassed)
  layout_same_as_defence: 'yes'
  measured_by: spotguard_applsci2026
  page: 19
  decoding: greedy, temperature 0
  sample_size: '250'
  baseline_kind: off_the_shelf
- id: ev_none_ben
  defence_id: none:locally hosted 7B-class open-weight model via Ollama (qwen2.5:7b serves the pipeline; the baseline model is not named separately)
  benchmark: InjecAgent-based stratified benchmark, 250 benign cases
  attacker_type: none
  attacker_access: not_applicable
  judge: rule-based automated judge (benign request served)
  layout_same_as_defence: 'yes'
  measured_by: spotguard_applsci2026
  page: 19
  decoding: greedy, temperature 0
  sample_size: '250'
  baseline_kind: off_the_shelf
- id: ev_spot
  defence_id: spot
  benchmark: InjecAgent-based stratified benchmark, 250 adversarial cases
  attacker_type: static
  attacker_access: black_box
  judge: rule-based automated judge (attack success when the defence is bypassed)
  layout_same_as_defence: 'yes'
  measured_by: spotguard_applsci2026
  page: 19
  decoding: greedy, temperature 0
  sample_size: '250'
- id: ev_spot_ben
  defence_id: spot
  benchmark: InjecAgent-based stratified benchmark, 250 benign cases
  attacker_type: none
  attacker_access: not_applicable
  judge: rule-based automated judge (benign request served)
  layout_same_as_defence: 'yes'
  measured_by: spotguard_applsci2026
  page: 19
  decoding: greedy, temperature 0
  sample_size: '250'
results:
- id: r_none_asr
  evaluation_id: ev_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '36.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 3
- id: r_none_bs
  evaluation_id: ev_none_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign success rate
  value: '93.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 3
- id: r_spot_asr
  evaluation_id: ev_spot
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '20.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 3
- id: r_spot_bs
  evaluation_id: ev_spot_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign success rate
  value: '98.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 3
comparisons:
- id: c_spot
  left: ev_spot
  right: ev_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 19
- id: c_spot_ben
  left: ev_spot_ben
  right: ev_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 19
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 3
    R2clean: Table 3 (benign success)
    R2attack: not reported
schema_gaps:
- where: report
  note: the paper does not name the model behind the Baseline and Spotlighting-Only arms or which spotlighting encoding they use (delimiters or Base64); its headline result (36.0% to 17.2%) is for the full layered system,
    which is out of scope; no claim is coded for the spotlighting arm alone

`advershield_jtsi2026` (doi_10_47134_jtsi_v3i2_6047)

In [ ]:
%%writefile reports/records/advershield_jtsi2026.yaml
report:
  id: advershield_jtsi2026
  slug: doi_10_47134_jtsi_v3i2_6047
  title: 'AdverShield-LLM: Adversarial Robustness Certification for IoT-Integrated Retrieval-Augmented Generation via Randomized Smoothing'
  first_author: Yasser Samir Hadi
  venue: Journal of Technology and System Information
  peer_reviewed: 'yes'
  doi: 10.47134/jtsi.v3i2.6047
  version_read: published version
  first_date: '2026-07-06'
  read: F
  role: proposes
  extraction_scope: Table 6 row TAGD only against Table 5 Vanilla RAG on MS-RAG-IoT (attack success, attack types pooled); the Table 6 remark on TAGD (the full system and PLSA isolate passages, query the model per group and
    aggregate, which is system level; RobustRAG, JailGuard, PoisonedRAG-Def, CAF-RAG and EBIDS+RAG are filters, detectors or system designs; the per-attack-type comparison is figure only)
defences:
- id: tagd
  name: TAGD only (token-adaptive Gaussian noise on input embeddings, scaled inversely to attention salience, with smoothed generation over 50 samples)
  origin: this_report
  base_checkpoint: Mistral-7B-Instruct-v0.3
  axis1:
  - internal_intervention
  axis2:
  - fixed_inference_rule
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 13
  layout: RAG prompt with retrieved IoT passages
  anchor: Token-Adaptive Gaussian Defense (TAGD)
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
evaluations:
- id: ev_none
  defence_id: none:Mistral-7B-Instruct-v0.3
  benchmark: MS-RAG-IoT, corpus poisoning, indirect prompt injection and embedding perturbation pooled
  attacker_type: static
  attacker_access: black_box
  judge: adversary's desired response produced
  layout_same_as_defence: 'yes'
  measured_by: advershield_jtsi2026
  page: 19
  decoding: not reported
  baseline_kind: off_the_shelf
- id: ev_tagd
  defence_id: tagd
  benchmark: MS-RAG-IoT, corpus poisoning, indirect prompt injection and embedding perturbation pooled
  attacker_type: static
  attacker_access: black_box
  judge: adversary's desired response produced
  layout_same_as_defence: 'yes'
  measured_by: advershield_jtsi2026
  page: 21
  decoding: not reported
results:
- id: r_none
  evaluation_id: ev_none
  quantity: R1
  coverage: proxy
  metric_as_printed: ASR, attack types pooled
  value: '74.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 5 (Vanilla RAG)
- id: r_tagd
  evaluation_id: ev_tagd
  quantity: R1
  coverage: proxy
  metric_as_printed: ASR, attack types pooled
  value: '31.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 21
  location: Table 6
comparisons:
- id: c_tagd
  left: ev_tagd
  right: ev_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched_unstated
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 21
claims:
- id: k1
  text: the generation-stage noise layer is needed against indirect prompt injection
  location: Section on the ablation study
  anchor: indicating the necessity of certification for generation stage
  jobs:
  - R1
  supported_by:
  - c_tagd
  page: 21
coverage:
  R1: proxy
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 5 and 6 (attack types pooled)
    R2clean: not reported for TAGD alone
    R2attack: not reported
schema_gaps:
- where: report
  note: attack success pools corpus poisoning, prompt injection and embedding perturbation, so it is a proxy for injection; the ablation setting (sigma 0.50, m=5) is not stated for Table 5; the 'PLSA only' row repeats the
    RobustRAG row of Table 5 exactly

`hybridrag_ijaiml2026` (doi_10_51483_ijaiml_2557)

In [ ]:
%%writefile reports/records/hybridrag_ijaiml2026.yaml
report:
  id: hybridrag_ijaiml2026
  slug: doi_10_51483_ijaiml_2557
  title: 'Hybrid Defense Against Indirect Prompt Injection in RAG: A Cross-Model Benchmark'
  first_author: Geetha Raghuraj
  venue: International Journal of Artificial Intelligence and Machine Learning
  peer_reviewed: 'yes'
  doi: 10.51483/ijaiml.2557
  version_read: published version (Vol. 6, No. 3)
  first_date: '2026-01-01'
  read: F
  role: proposes
  extraction_scope: 'Tables I and II: None, Sanitize, Instruction Hierarchy and Hybrid for three models (ASR on attacked queries, accuracy on clean queries); the abstract claim (Output Verification passes the answer to a second
    checking call and is left out as a detector)'
defences:
- id: san_qw
  name: Sanitize (retrieved content in <retrieved_data> tags with a system-level note that it is untrusted)
  origin: other_report
  base_checkpoint: Qwen2.5-7B-Instruct (4-bit)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: system prompt and user prompt with the top-3 retrieved passages and the question
  layout_compat: &id001
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: Explicit <retrieved_data> delimiter tags are used to enclose retrieved content
- id: ih_qw
  name: Instruction Hierarchy prompt (system > user task > retrieved context; ignore embedded commands)
  origin: other_report
  base_checkpoint: Qwen2.5-7B-Instruct (4-bit)
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: system prompt and user prompt with the top-3 retrieved passages and the question
  layout_compat: *id001
  anchor: A stronger system prompt sets an explicit priority order
- id: hyb_qw
  name: Hybrid (Sanitize tags plus the instruction-hierarchy system prompt)
  origin: this_report
  base_checkpoint: Qwen2.5-7B-Instruct (4-bit)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: system prompt and user prompt with the top-3 retrieved passages and the question
  layout_compat: *id001
  anchor: 'Boundary layer: C is wrapped as'
- id: san_phi
  name: Sanitize (retrieved content in <retrieved_data> tags with a system-level note that it is untrusted)
  origin: other_report
  base_checkpoint: Phi-3-mini-4k-instruct (4-bit)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: system prompt and user prompt with the top-3 retrieved passages and the question
  layout_compat: *id001
  anchor: Explicit <retrieved_data> delimiter tags are used to enclose retrieved content
- id: ih_phi
  name: Instruction Hierarchy prompt (system > user task > retrieved context; ignore embedded commands)
  origin: other_report
  base_checkpoint: Phi-3-mini-4k-instruct (4-bit)
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: system prompt and user prompt with the top-3 retrieved passages and the question
  layout_compat: *id001
  anchor: A stronger system prompt sets an explicit priority order
- id: hyb_phi
  name: Hybrid (Sanitize tags plus the instruction-hierarchy system prompt)
  origin: this_report
  base_checkpoint: Phi-3-mini-4k-instruct (4-bit)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: system prompt and user prompt with the top-3 retrieved passages and the question
  layout_compat: *id001
  anchor: 'Boundary layer: C is wrapped as'
- id: san_mis
  name: Sanitize (retrieved content in <retrieved_data> tags with a system-level note that it is untrusted)
  origin: other_report
  base_checkpoint: Mistral-7B-Instruct-v0.3 (4-bit)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: system prompt and user prompt with the top-3 retrieved passages and the question
  layout_compat: *id001
  anchor: Explicit <retrieved_data> delimiter tags are used to enclose retrieved content
- id: ih_mis
  name: Instruction Hierarchy prompt (system > user task > retrieved context; ignore embedded commands)
  origin: other_report
  base_checkpoint: Mistral-7B-Instruct-v0.3 (4-bit)
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: system prompt and user prompt with the top-3 retrieved passages and the question
  layout_compat: *id001
  anchor: A stronger system prompt sets an explicit priority order
- id: hyb_mis
  name: Hybrid (Sanitize tags plus the instruction-hierarchy system prompt)
  origin: this_report
  base_checkpoint: Mistral-7B-Instruct-v0.3 (4-bit)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: system prompt and user prompt with the top-3 retrieved passages and the question
  layout_compat: *id001
  anchor: 'Boundary layer: C is wrapped as'
evaluations:
- id: ev_qw_none
  defence_id: none:Qwen2.5-7B-Instruct (4-bit)
  benchmark: SQuAD 60-question RAG corpus, 35% of passages poisoned (about 20 attacked queries)
  attacker_type: static
  attacker_access: black_box
  judge: attack canary string in the output
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 6
  decoding: deterministic
  sample_size: about 20 attacked queries
  baseline_kind: instruction_prompt_only
- id: ev_qw_none_clean
  defence_id: none:Qwen2.5-7B-Instruct (4-bit)
  benchmark: SQuAD 60-question RAG corpus, clean queries (about 40)
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 7
  decoding: deterministic
  sample_size: about 40 clean queries
  baseline_kind: instruction_prompt_only
- id: ev_qw_san
  defence_id: san_qw
  benchmark: SQuAD 60-question RAG corpus, 35% of passages poisoned (about 20 attacked queries)
  attacker_type: static
  attacker_access: black_box
  judge: attack canary string in the output
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 6
  decoding: deterministic
  sample_size: about 20 attacked queries
- id: ev_qw_san_clean
  defence_id: san_qw
  benchmark: SQuAD 60-question RAG corpus, clean queries (about 40)
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 7
  decoding: deterministic
  sample_size: about 40 clean queries
- id: ev_qw_ih
  defence_id: ih_qw
  benchmark: SQuAD 60-question RAG corpus, 35% of passages poisoned (about 20 attacked queries)
  attacker_type: static
  attacker_access: black_box
  judge: attack canary string in the output
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 6
  decoding: deterministic
  sample_size: about 20 attacked queries
- id: ev_qw_ih_clean
  defence_id: ih_qw
  benchmark: SQuAD 60-question RAG corpus, clean queries (about 40)
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 7
  decoding: deterministic
  sample_size: about 40 clean queries
- id: ev_qw_hyb
  defence_id: hyb_qw
  benchmark: SQuAD 60-question RAG corpus, 35% of passages poisoned (about 20 attacked queries)
  attacker_type: static
  attacker_access: black_box
  judge: attack canary string in the output
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 6
  decoding: deterministic
  sample_size: about 20 attacked queries
- id: ev_qw_hyb_clean
  defence_id: hyb_qw
  benchmark: SQuAD 60-question RAG corpus, clean queries (about 40)
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 7
  decoding: deterministic
  sample_size: about 40 clean queries
- id: ev_phi_none
  defence_id: none:Phi-3-mini-4k-instruct (4-bit)
  benchmark: SQuAD 60-question RAG corpus, 35% of passages poisoned (about 20 attacked queries)
  attacker_type: static
  attacker_access: black_box
  judge: attack canary string in the output
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 6
  decoding: deterministic
  sample_size: about 20 attacked queries
  baseline_kind: instruction_prompt_only
- id: ev_phi_none_clean
  defence_id: none:Phi-3-mini-4k-instruct (4-bit)
  benchmark: SQuAD 60-question RAG corpus, clean queries (about 40)
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 7
  decoding: deterministic
  sample_size: about 40 clean queries
  baseline_kind: instruction_prompt_only
- id: ev_phi_san
  defence_id: san_phi
  benchmark: SQuAD 60-question RAG corpus, 35% of passages poisoned (about 20 attacked queries)
  attacker_type: static
  attacker_access: black_box
  judge: attack canary string in the output
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 6
  decoding: deterministic
  sample_size: about 20 attacked queries
- id: ev_phi_san_clean
  defence_id: san_phi
  benchmark: SQuAD 60-question RAG corpus, clean queries (about 40)
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 7
  decoding: deterministic
  sample_size: about 40 clean queries
- id: ev_phi_ih
  defence_id: ih_phi
  benchmark: SQuAD 60-question RAG corpus, 35% of passages poisoned (about 20 attacked queries)
  attacker_type: static
  attacker_access: black_box
  judge: attack canary string in the output
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 6
  decoding: deterministic
  sample_size: about 20 attacked queries
- id: ev_phi_ih_clean
  defence_id: ih_phi
  benchmark: SQuAD 60-question RAG corpus, clean queries (about 40)
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 7
  decoding: deterministic
  sample_size: about 40 clean queries
- id: ev_phi_hyb
  defence_id: hyb_phi
  benchmark: SQuAD 60-question RAG corpus, 35% of passages poisoned (about 20 attacked queries)
  attacker_type: static
  attacker_access: black_box
  judge: attack canary string in the output
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 6
  decoding: deterministic
  sample_size: about 20 attacked queries
- id: ev_phi_hyb_clean
  defence_id: hyb_phi
  benchmark: SQuAD 60-question RAG corpus, clean queries (about 40)
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 7
  decoding: deterministic
  sample_size: about 40 clean queries
- id: ev_mis_none
  defence_id: none:Mistral-7B-Instruct-v0.3 (4-bit)
  benchmark: SQuAD 60-question RAG corpus, 35% of passages poisoned (about 20 attacked queries)
  attacker_type: static
  attacker_access: black_box
  judge: attack canary string in the output
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 6
  decoding: deterministic
  sample_size: about 20 attacked queries
  baseline_kind: instruction_prompt_only
- id: ev_mis_none_clean
  defence_id: none:Mistral-7B-Instruct-v0.3 (4-bit)
  benchmark: SQuAD 60-question RAG corpus, clean queries (about 40)
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 7
  decoding: deterministic
  sample_size: about 40 clean queries
  baseline_kind: instruction_prompt_only
- id: ev_mis_san
  defence_id: san_mis
  benchmark: SQuAD 60-question RAG corpus, 35% of passages poisoned (about 20 attacked queries)
  attacker_type: static
  attacker_access: black_box
  judge: attack canary string in the output
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 6
  decoding: deterministic
  sample_size: about 20 attacked queries
- id: ev_mis_san_clean
  defence_id: san_mis
  benchmark: SQuAD 60-question RAG corpus, clean queries (about 40)
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 7
  decoding: deterministic
  sample_size: about 40 clean queries
- id: ev_mis_ih
  defence_id: ih_mis
  benchmark: SQuAD 60-question RAG corpus, 35% of passages poisoned (about 20 attacked queries)
  attacker_type: static
  attacker_access: black_box
  judge: attack canary string in the output
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 6
  decoding: deterministic
  sample_size: about 20 attacked queries
- id: ev_mis_ih_clean
  defence_id: ih_mis
  benchmark: SQuAD 60-question RAG corpus, clean queries (about 40)
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 7
  decoding: deterministic
  sample_size: about 40 clean queries
- id: ev_mis_hyb
  defence_id: hyb_mis
  benchmark: SQuAD 60-question RAG corpus, 35% of passages poisoned (about 20 attacked queries)
  attacker_type: static
  attacker_access: black_box
  judge: attack canary string in the output
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 6
  decoding: deterministic
  sample_size: about 20 attacked queries
- id: ev_mis_hyb_clean
  defence_id: hyb_mis
  benchmark: SQuAD 60-question RAG corpus, clean queries (about 40)
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: hybridrag_ijaiml2026
  page: 7
  decoding: deterministic
  sample_size: about 40 clean queries
results:
- id: r_qw_none_asr
  evaluation_id: ev_qw_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table I
- id: r_qw_none_acc
  evaluation_id: ev_qw_none_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: task accuracy on clean queries
  value: '72.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table II
- id: r_qw_san_asr
  evaluation_id: ev_qw_san
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '15'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table I
- id: r_qw_san_acc
  evaluation_id: ev_qw_san_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: task accuracy on clean queries
  value: '82.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table II
- id: r_qw_ih_asr
  evaluation_id: ev_qw_ih
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table I
- id: r_qw_ih_acc
  evaluation_id: ev_qw_ih_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: task accuracy on clean queries
  value: '77.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table II
- id: r_qw_hyb_asr
  evaluation_id: ev_qw_hyb
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '10'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table I
- id: r_qw_hyb_acc
  evaluation_id: ev_qw_hyb_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: task accuracy on clean queries
  value: '85.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table II
- id: r_phi_none_asr
  evaluation_id: ev_phi_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '10'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table I
- id: r_phi_none_acc
  evaluation_id: ev_phi_none_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: task accuracy on clean queries
  value: '72.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table II
- id: r_phi_san_asr
  evaluation_id: ev_phi_san
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table I
- id: r_phi_san_acc
  evaluation_id: ev_phi_san_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: task accuracy on clean queries
  value: '77.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table II
- id: r_phi_ih_asr
  evaluation_id: ev_phi_ih
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table I
- id: r_phi_ih_acc
  evaluation_id: ev_phi_ih_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: task accuracy on clean queries
  value: '77.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table II
- id: r_phi_hyb_asr
  evaluation_id: ev_phi_hyb
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table I
- id: r_phi_hyb_acc
  evaluation_id: ev_phi_hyb_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: task accuracy on clean queries
  value: '77.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table II
- id: r_mis_none_asr
  evaluation_id: ev_mis_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '55'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table I
- id: r_mis_none_acc
  evaluation_id: ev_mis_none_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: task accuracy on clean queries
  value: '67.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table II
- id: r_mis_san_asr
  evaluation_id: ev_mis_san
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table I
- id: r_mis_san_acc
  evaluation_id: ev_mis_san_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: task accuracy on clean queries
  value: '67.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table II
- id: r_mis_ih_asr
  evaluation_id: ev_mis_ih
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '25'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table I
- id: r_mis_ih_acc
  evaluation_id: ev_mis_ih_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: task accuracy on clean queries
  value: '70.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table II
- id: r_mis_hyb_asr
  evaluation_id: ev_mis_hyb
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '10'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table I
- id: r_mis_hyb_acc
  evaluation_id: ev_mis_hyb_clean
  quantity: R2clean
  coverage: direct
  metric_as_printed: task accuracy on clean queries
  value: '65.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table II
comparisons:
- id: c_qw_san
  left: ev_qw_san
  right: ev_qw_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_qw_san_clean
  left: ev_qw_san_clean
  right: ev_qw_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_qw_ih
  left: ev_qw_ih
  right: ev_qw_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_qw_ih_clean
  left: ev_qw_ih_clean
  right: ev_qw_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_qw_hyb
  left: ev_qw_hyb
  right: ev_qw_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_qw_hyb_clean
  left: ev_qw_hyb_clean
  right: ev_qw_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_qw_hyb_vs_san
  left: ev_qw_hyb
  right: ev_qw_san
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the hybrid against its Sanitize component alone
  page: 6
- id: c_qw_hyb_vs_ih
  left: ev_qw_hyb
  right: ev_qw_ih
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the hybrid against its instruction-hierarchy component alone
  page: 6
- id: c_phi_san
  left: ev_phi_san
  right: ev_phi_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_phi_san_clean
  left: ev_phi_san_clean
  right: ev_phi_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_phi_ih
  left: ev_phi_ih
  right: ev_phi_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_phi_ih_clean
  left: ev_phi_ih_clean
  right: ev_phi_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_phi_hyb
  left: ev_phi_hyb
  right: ev_phi_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_phi_hyb_clean
  left: ev_phi_hyb_clean
  right: ev_phi_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_phi_hyb_vs_san
  left: ev_phi_hyb
  right: ev_phi_san
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the hybrid against its Sanitize component alone
  page: 6
- id: c_phi_hyb_vs_ih
  left: ev_phi_hyb
  right: ev_phi_ih
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the hybrid against its instruction-hierarchy component alone
  page: 6
- id: c_mis_san
  left: ev_mis_san
  right: ev_mis_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_mis_san_clean
  left: ev_mis_san_clean
  right: ev_mis_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_mis_ih
  left: ev_mis_ih
  right: ev_mis_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_mis_ih_clean
  left: ev_mis_ih_clean
  right: ev_mis_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_mis_hyb
  left: ev_mis_hyb
  right: ev_mis_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_mis_hyb_clean
  left: ev_mis_hyb_clean
  right: ev_mis_none_clean
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_mis_hyb_vs_san
  left: ev_mis_hyb
  right: ev_mis_san
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the hybrid against its Sanitize component alone
  page: 6
- id: c_mis_hyb_vs_ih
  left: ev_mis_hyb
  right: ev_mis_ih
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the hybrid against its instruction-hierarchy component alone
  page: 6
claims:
- id: k1
  text: the hybrid defence never showed a severe accuracy drop in any model
  location: abstract
  anchor: the hybrid defense never showed any severe accuracy drop in any model
  jobs:
  - R1
  - R2
  supported_by:
  - c_qw_hyb
  - c_phi_hyb
  - c_mis_hyb
  - c_qw_hyb_clean
  - c_phi_hyb_clean
  - c_mis_hyb_clean
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table I
    R2clean: Table II
    R2attack: not reported
schema_gaps:
- where: report
  note: about 20 attacked and 40 clean queries per model, so a single query moves ASR by 5 points; on Mistral the hybrid is above Sanitize alone (10% vs 5%)

`eroded_zenodo2026` (doi_10_5281_zenodo_23972864)

In [ ]:
%%writefile reports/records/eroded_zenodo2026.yaml
report:
  id: eroded_zenodo2026
  slug: doi_10_5281_zenodo_23972864
  title: 'Code, data and extended report for "Defended in Training, Eroded in Deployment: Benign Fine-Tuning Undermines Weight-Based Prompt-Injection Defenses"'
  first_author: Hai Ho
  venue: Zenodo (extended report with code and data)
  peer_reviewed: 'no'
  doi: 10.5281/zenodo.23972864
  version_read: Zenodo record
  first_date: '2026-11-05'
  read: F
  role: re_measures
  extraction_scope: 'Table E4: undefended Llama-3.1-8B-Instruct and merged Meta-SecAlign-8B, each with and without benign QLoRA fine-tuning (three seeds), the SFT repair variants, worst-case and per-attack ASR on six static
    attacks and MMLU; the Figure E4 claim (the figures E1 to E6 repeat or extend supplementary tables of the main paper, which is not this record; quantisation rows are not defences)'
defences:
- id: msa
  name: Meta-SecAlign-8B, merged BF16, null adapter
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 1
  layout: AlpacaFarm instruction with the data in the input role (Meta-SecAlign template)
  layout_compat: &id001
    user_turn: untested
    separate_role: supported
    system_context: untested
    agent_tool_result: untested
  anchor: Meta-SecAlign, merged BF16
- id: msaft2026
  name: Meta-SecAlign-8B merged, then benign QLoRA fine-tuning on 5,000 Alpaca examples (seed 2026)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  - sft
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  layout: AlpacaFarm instruction with the data in the input role (Meta-SecAlign template)
  layout_compat: *id001
  anchor: Meta-SecAlign merged + FT s2026
- id: ftrep2026
  name: SFT repair on security examples applied to the benignly fine-tuned undefended model (seed 2026)
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct + benign fine-tuning (seed 2026)
  axis1:
  - chat_roles
  axis2:
  - sft
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  layout: AlpacaFarm instruction with the data in the input role (Meta-SecAlign template)
  layout_compat: *id001
  anchor: Undefended + FT + repair s2026
- id: msaft2027
  name: Meta-SecAlign-8B merged, then benign QLoRA fine-tuning on 5,000 Alpaca examples (seed 2027)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  - sft
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  layout: AlpacaFarm instruction with the data in the input role (Meta-SecAlign template)
  layout_compat: *id001
  anchor: Meta-SecAlign merged + FT s2027
- id: ftrep2027
  name: SFT repair on security examples applied to the benignly fine-tuned undefended model (seed 2027)
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct + benign fine-tuning (seed 2027)
  axis1:
  - chat_roles
  axis2:
  - sft
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  layout: AlpacaFarm instruction with the data in the input role (Meta-SecAlign template)
  layout_compat: *id001
  anchor: Undefended + FT + repair s2027
- id: msaft2028
  name: Meta-SecAlign-8B merged, then benign QLoRA fine-tuning on 5,000 Alpaca examples (seed 2028)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  - sft
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  layout: AlpacaFarm instruction with the data in the input role (Meta-SecAlign template)
  layout_compat: *id001
  anchor: Meta-SecAlign merged + FT s2028
- id: ftrep2028
  name: SFT repair on security examples applied to the benignly fine-tuned undefended model (seed 2028)
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct + benign fine-tuning (seed 2028)
  axis1:
  - chat_roles
  axis2:
  - sft
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  layout: AlpacaFarm instruction with the data in the input role (Meta-SecAlign template)
  layout_compat: *id001
  anchor: Undefended + FT + repair s2028
- id: reponly
  name: SFT repair on security examples applied to the undefended model
  origin: this_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - sft
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  layout: AlpacaFarm instruction with the data in the input role (Meta-SecAlign template)
  layout_compat: *id001
  anchor: Undefended + repair only
evaluations:
- id: ev_undef
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: AlpacaFarm, 208 samples, six static injection attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack output check (worst case: any of the six attacks succeeds)'
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '208'
  baseline_kind: off_the_shelf
- id: ev_undef_mmlu
  defence_id: none:Llama-3.1-8B-Instruct
  benchmark: MMLU, 300 questions
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '300'
  baseline_kind: off_the_shelf
- id: ev_undft2026
  defence_id: none:Llama-3.1-8B-Instruct + benign fine-tuning (seed 2026)
  benchmark: AlpacaFarm, 208 samples, six static injection attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack output check (worst case: any of the six attacks succeeds)'
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '208'
  baseline_kind: same_data_twin
- id: ev_undft2026_mmlu
  defence_id: none:Llama-3.1-8B-Instruct + benign fine-tuning (seed 2026)
  benchmark: MMLU, 300 questions
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '300'
  baseline_kind: same_data_twin
- id: ev_undft2027
  defence_id: none:Llama-3.1-8B-Instruct + benign fine-tuning (seed 2027)
  benchmark: AlpacaFarm, 208 samples, six static injection attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack output check (worst case: any of the six attacks succeeds)'
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '208'
  baseline_kind: same_data_twin
- id: ev_undft2027_mmlu
  defence_id: none:Llama-3.1-8B-Instruct + benign fine-tuning (seed 2027)
  benchmark: MMLU, 300 questions
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '300'
  baseline_kind: same_data_twin
- id: ev_undft2028
  defence_id: none:Llama-3.1-8B-Instruct + benign fine-tuning (seed 2028)
  benchmark: AlpacaFarm, 208 samples, six static injection attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack output check (worst case: any of the six attacks succeeds)'
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '208'
  baseline_kind: same_data_twin
- id: ev_undft2028_mmlu
  defence_id: none:Llama-3.1-8B-Instruct + benign fine-tuning (seed 2028)
  benchmark: MMLU, 300 questions
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '300'
  baseline_kind: same_data_twin
- id: ev_ftrep2026
  defence_id: ftrep2026
  benchmark: AlpacaFarm, 208 samples, six static injection attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack output check (worst case: any of the six attacks succeeds)'
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '208'
- id: ev_ftrep2026_mmlu
  defence_id: ftrep2026
  benchmark: MMLU, 300 questions
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '300'
- id: ev_ftrep2027
  defence_id: ftrep2027
  benchmark: AlpacaFarm, 208 samples, six static injection attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack output check (worst case: any of the six attacks succeeds)'
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '208'
- id: ev_ftrep2027_mmlu
  defence_id: ftrep2027
  benchmark: MMLU, 300 questions
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '300'
- id: ev_ftrep2028
  defence_id: ftrep2028
  benchmark: AlpacaFarm, 208 samples, six static injection attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack output check (worst case: any of the six attacks succeeds)'
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '208'
- id: ev_ftrep2028_mmlu
  defence_id: ftrep2028
  benchmark: MMLU, 300 questions
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '300'
- id: ev_reponly
  defence_id: reponly
  benchmark: AlpacaFarm, 208 samples, six static injection attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack output check (worst case: any of the six attacks succeeds)'
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '208'
- id: ev_reponly_mmlu
  defence_id: reponly
  benchmark: MMLU, 300 questions
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '300'
- id: ev_msa
  defence_id: msa
  benchmark: AlpacaFarm, 208 samples, six static injection attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack output check (worst case: any of the six attacks succeeds)'
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '208'
- id: ev_msa_mmlu
  defence_id: msa
  benchmark: MMLU, 300 questions
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '300'
- id: ev_msaft2026
  defence_id: msaft2026
  benchmark: AlpacaFarm, 208 samples, six static injection attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack output check (worst case: any of the six attacks succeeds)'
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '208'
- id: ev_msaft2026_mmlu
  defence_id: msaft2026
  benchmark: MMLU, 300 questions
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '300'
- id: ev_msaft2027
  defence_id: msaft2027
  benchmark: AlpacaFarm, 208 samples, six static injection attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack output check (worst case: any of the six attacks succeeds)'
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '208'
- id: ev_msaft2027_mmlu
  defence_id: msaft2027
  benchmark: MMLU, 300 questions
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '300'
- id: ev_msaft2028
  defence_id: msaft2028
  benchmark: AlpacaFarm, 208 samples, six static injection attacks
  attacker_type: static
  attacker_access: black_box
  judge: 'attack output check (worst case: any of the six attacks succeeds)'
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '208'
- id: ev_msaft2028_mmlu
  defence_id: msaft2028
  benchmark: MMLU, 300 questions
  attacker_type: none
  attacker_access: not_applicable
  judge: answer correct
  layout_same_as_defence: 'yes'
  measured_by: eroded_zenodo2026
  page: 4
  decoding: not reported
  sample_size: '300'
results:
- id: r_undef_worst_case
  evaluation_id: ev_undef
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, worst case
  value: '91.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undef_naive
  evaluation_id: ev_undef
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, naive
  value: '7.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undef_ignore
  evaluation_id: ev_undef
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, ignore
  value: '9.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undef_escape
  evaluation_id: ev_undef
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, escape
  value: '10.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undef_fake_completion
  evaluation_id: ev_undef
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake completion
  value: '33.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undef_fake_role_header
  evaluation_id: ev_undef
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake role header
  value: '87.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undef_combined
  evaluation_id: ev_undef
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, combined
  value: '57.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undef_mmlu
  evaluation_id: ev_undef_mmlu
  quantity: R2clean
  coverage: direct
  metric_as_printed: MMLU accuracy
  value: '63.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2026_worst_case
  evaluation_id: ev_undft2026
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, worst case
  value: '96.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2026_naive
  evaluation_id: ev_undft2026
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, naive
  value: '9.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2026_ignore
  evaluation_id: ev_undft2026
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, ignore
  value: '37.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2026_escape
  evaluation_id: ev_undft2026
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, escape
  value: '23.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2026_fake_completion
  evaluation_id: ev_undft2026
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake completion
  value: '85.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2026_fake_role_header
  evaluation_id: ev_undft2026
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake role header
  value: '91.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2026_combined
  evaluation_id: ev_undft2026
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, combined
  value: '90.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2026_mmlu
  evaluation_id: ev_undft2026_mmlu
  quantity: R2clean
  coverage: direct
  metric_as_printed: MMLU accuracy
  value: '57.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2027_worst_case
  evaluation_id: ev_undft2027
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, worst case
  value: '98.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2027_naive
  evaluation_id: ev_undft2027
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, naive
  value: '9.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2027_ignore
  evaluation_id: ev_undft2027
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, ignore
  value: '35.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2027_escape
  evaluation_id: ev_undft2027
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, escape
  value: '23.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2027_fake_completion
  evaluation_id: ev_undft2027
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake completion
  value: '90.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2027_fake_role_header
  evaluation_id: ev_undft2027
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake role header
  value: '95.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2027_combined
  evaluation_id: ev_undft2027
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, combined
  value: '96.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2027_mmlu
  evaluation_id: ev_undft2027_mmlu
  quantity: R2clean
  coverage: direct
  metric_as_printed: MMLU accuracy
  value: '62.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2028_worst_case
  evaluation_id: ev_undft2028
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, worst case
  value: '98.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2028_naive
  evaluation_id: ev_undft2028
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, naive
  value: '26.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2028_ignore
  evaluation_id: ev_undft2028
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, ignore
  value: '55.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2028_escape
  evaluation_id: ev_undft2028
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, escape
  value: '29.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2028_fake_completion
  evaluation_id: ev_undft2028
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake completion
  value: '92.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2028_fake_role_header
  evaluation_id: ev_undft2028
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake role header
  value: '94.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2028_combined
  evaluation_id: ev_undft2028
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, combined
  value: '95.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_undft2028_mmlu
  evaluation_id: ev_undft2028_mmlu
  quantity: R2clean
  coverage: direct
  metric_as_printed: MMLU accuracy
  value: '63.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2026_worst_case
  evaluation_id: ev_ftrep2026
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, worst case
  value: '3.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2026_naive
  evaluation_id: ev_ftrep2026
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, naive
  value: '1.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2026_ignore
  evaluation_id: ev_ftrep2026
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, ignore
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2026_escape
  evaluation_id: ev_ftrep2026
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, escape
  value: '3.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2026_fake_completion
  evaluation_id: ev_ftrep2026
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake completion
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2026_fake_role_header
  evaluation_id: ev_ftrep2026
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake role header
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2026_combined
  evaluation_id: ev_ftrep2026
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, combined
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2026_mmlu
  evaluation_id: ev_ftrep2026_mmlu
  quantity: R2clean
  coverage: direct
  metric_as_printed: MMLU accuracy
  value: '58.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2027_worst_case
  evaluation_id: ev_ftrep2027
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, worst case
  value: '2.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2027_naive
  evaluation_id: ev_ftrep2027
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, naive
  value: '1.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2027_ignore
  evaluation_id: ev_ftrep2027
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, ignore
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2027_escape
  evaluation_id: ev_ftrep2027
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, escape
  value: '2.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2027_fake_completion
  evaluation_id: ev_ftrep2027
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake completion
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2027_fake_role_header
  evaluation_id: ev_ftrep2027
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake role header
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2027_combined
  evaluation_id: ev_ftrep2027
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, combined
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2027_mmlu
  evaluation_id: ev_ftrep2027_mmlu
  quantity: R2clean
  coverage: direct
  metric_as_printed: MMLU accuracy
  value: '61.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2028_worst_case
  evaluation_id: ev_ftrep2028
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, worst case
  value: '3.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2028_naive
  evaluation_id: ev_ftrep2028
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, naive
  value: '3.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2028_ignore
  evaluation_id: ev_ftrep2028
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, ignore
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2028_escape
  evaluation_id: ev_ftrep2028
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, escape
  value: '1.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2028_fake_completion
  evaluation_id: ev_ftrep2028
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake completion
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2028_fake_role_header
  evaluation_id: ev_ftrep2028
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake role header
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2028_combined
  evaluation_id: ev_ftrep2028
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, combined
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_ftrep2028_mmlu
  evaluation_id: ev_ftrep2028_mmlu
  quantity: R2clean
  coverage: direct
  metric_as_printed: MMLU accuracy
  value: '60.7'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_reponly_worst_case
  evaluation_id: ev_reponly
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, worst case
  value: '12.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_reponly_naive
  evaluation_id: ev_reponly
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, naive
  value: '10.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_reponly_ignore
  evaluation_id: ev_reponly
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, ignore
  value: '6.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_reponly_escape
  evaluation_id: ev_reponly
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, escape
  value: '7.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_reponly_fake_completion
  evaluation_id: ev_reponly
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake completion
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_reponly_fake_role_header
  evaluation_id: ev_reponly
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake role header
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_reponly_combined
  evaluation_id: ev_reponly
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, combined
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_reponly_mmlu
  evaluation_id: ev_reponly_mmlu
  quantity: R2clean
  coverage: direct
  metric_as_printed: MMLU accuracy
  value: '62.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msa_worst_case
  evaluation_id: ev_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, worst case
  value: '3.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msa_naive
  evaluation_id: ev_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, naive
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msa_ignore
  evaluation_id: ev_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, ignore
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msa_escape
  evaluation_id: ev_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, escape
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msa_fake_completion
  evaluation_id: ev_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake completion
  value: '0.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msa_fake_role_header
  evaluation_id: ev_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake role header
  value: '3.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msa_combined
  evaluation_id: ev_msa
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, combined
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msa_mmlu
  evaluation_id: ev_msa_mmlu
  quantity: R2clean
  coverage: direct
  metric_as_printed: MMLU accuracy
  value: '62.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2026_worst_case
  evaluation_id: ev_msaft2026
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, worst case
  value: '82.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2026_naive
  evaluation_id: ev_msaft2026
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, naive
  value: '3.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2026_ignore
  evaluation_id: ev_msaft2026
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, ignore
  value: '13.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2026_escape
  evaluation_id: ev_msaft2026
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, escape
  value: '8.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2026_fake_completion
  evaluation_id: ev_msaft2026
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake completion
  value: '31.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2026_fake_role_header
  evaluation_id: ev_msaft2026
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake role header
  value: '76.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2026_combined
  evaluation_id: ev_msaft2026
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, combined
  value: '33.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2026_mmlu
  evaluation_id: ev_msaft2026_mmlu
  quantity: R2clean
  coverage: direct
  metric_as_printed: MMLU accuracy
  value: '57.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2027_worst_case
  evaluation_id: ev_msaft2027
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, worst case
  value: '86.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2027_naive
  evaluation_id: ev_msaft2027
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, naive
  value: '5.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2027_ignore
  evaluation_id: ev_msaft2027
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, ignore
  value: '13.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2027_escape
  evaluation_id: ev_msaft2027
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, escape
  value: '4.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2027_fake_completion
  evaluation_id: ev_msaft2027
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake completion
  value: '54.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2027_fake_role_header
  evaluation_id: ev_msaft2027
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake role header
  value: '78.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2027_combined
  evaluation_id: ev_msaft2027
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, combined
  value: '57.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2027_mmlu
  evaluation_id: ev_msaft2027_mmlu
  quantity: R2clean
  coverage: direct
  metric_as_printed: MMLU accuracy
  value: '61.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2028_worst_case
  evaluation_id: ev_msaft2028
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, worst case
  value: '92.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2028_naive
  evaluation_id: ev_msaft2028
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, naive
  value: '12.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2028_ignore
  evaluation_id: ev_msaft2028
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, ignore
  value: '18.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2028_escape
  evaluation_id: ev_msaft2028
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, escape
  value: '10.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2028_fake_completion
  evaluation_id: ev_msaft2028
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake completion
  value: '63.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2028_fake_role_header
  evaluation_id: ev_msaft2028
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fake role header
  value: '88.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2028_combined
  evaluation_id: ev_msaft2028
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, combined
  value: '71.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
- id: r_msaft2028_mmlu
  evaluation_id: ev_msaft2028_mmlu
  quantity: R2clean
  coverage: direct
  metric_as_printed: MMLU accuracy
  value: '63.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table E4
comparisons:
- id: c_msa
  left: ev_msa
  right: ev_undef
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_msa_mmlu
  left: ev_msa_mmlu
  right: ev_undef_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_reponly
  left: ev_reponly
  right: ev_undef
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_reponly_mmlu
  left: ev_reponly_mmlu
  right: ev_undef_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_ftrep2026
  left: ev_ftrep2026
  right: ev_undft2026
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_ftrep2026_mmlu
  left: ev_ftrep2026_mmlu
  right: ev_undft2026_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_msaft2026_vs_msa
  left: ev_msaft2026
  right: ev_msa
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: benign fine-tuning applied after the defence
  page: 4
- id: c_msaft2026_vs_undft
  left: ev_msaft2026
  right: ev_undft2026
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence present before the same benign fine-tuning
  page: 4
- id: c_ftrep2027
  left: ev_ftrep2027
  right: ev_undft2027
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_ftrep2027_mmlu
  left: ev_ftrep2027_mmlu
  right: ev_undft2027_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_msaft2027_vs_msa
  left: ev_msaft2027
  right: ev_msa
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: benign fine-tuning applied after the defence
  page: 4
- id: c_msaft2027_vs_undft
  left: ev_msaft2027
  right: ev_undft2027
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence present before the same benign fine-tuning
  page: 4
- id: c_ftrep2028
  left: ev_ftrep2028
  right: ev_undft2028
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_ftrep2028_mmlu
  left: ev_ftrep2028_mmlu
  right: ev_undft2028_mmlu
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 4
- id: c_msaft2028_vs_msa
  left: ev_msaft2028
  right: ev_msa
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: benign fine-tuning applied after the defence
  page: 4
- id: c_msaft2028_vs_undft
  left: ev_msaft2028
  right: ev_undft2028
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence present before the same benign fine-tuning
  page: 4
claims:
- id: k1
  text: benign fine-tuning on ordinary instruction data removes weight-based prompt-injection defences
  location: Figure E4 caption
  anchor: Benign fine-tuning (orange) removes all three defenses at n = 5000
  jobs:
  - R1
  supported_by:
  - c_msaft2026_vs_msa
  - c_msaft2027_vs_msa
  - c_msaft2028_vs_msa
  page: 2
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table E4
    R2clean: Table E4 (MMLU, 300 questions)
    R2attack: not reported
schema_gaps:
- where: report
  note: this record is the extended report archived with the code and data; its Table E4 comes from the main paper's merge-path setting; the attacks are the six static templates, with the worst case over them reported first

`municipal_sauc2026` (doi_10_62161_sauc_v12_6399)

In [ ]:
%%writefile reports/records/municipal_sauc2026.yaml
report:
  id: municipal_sauc2026
  slug: doi_10_62161_sauc_v12_6399
  title: 'Indirect Prompt Injection in Municipal Document-Processing Copilots: Attacks, Defences and Harm'
  first_author: Jorge Cisneros-González
  venue: Street Art & Urban Creativity
  peer_reviewed: 'yes'
  doi: 10.62161/sauc.v12.6399
  version_read: published version (12(5), 2026)
  first_date: '2026-09-05'
  read: F
  role: re_measures
  extraction_scope: 'Tables 2 to 4: no defence (D0), datamarking (D1) and structured-query delimiting with an instruction hierarchy (D2), pooled over four models, ASR by objective and overall and fidelity on legitimate cases;
    the abstract claim (the guardrail D3 screens untrusted spans and is a detector; the combined condition D4 contains it)'
defences:
- id: dm
  name: D1 spotlighting by datamarking of untrusted spans
  origin: other_report
  base_checkpoint: DeepSeek-V4-Flash, Qwen3.7-flash, Gemini 3.5 Flash-Lite and Mistral Small, pooled
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: system prompt with criteria and task; case file serialised as JSON in the prompt; payloads in free text, fields, attachments, hidden text or tables
  layout_compat: &id001
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: Spotlighting through datamarking supplies the model
- id: sq
  name: D2 structured-query delimiting with an instruction hierarchy (prompt level)
  origin: other_report
  base_checkpoint: DeepSeek-V4-Flash, Qwen3.7-flash, Gemini 3.5 Flash-Lite and Mistral Small, pooled
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 5
  layout: system prompt with criteria and task; case file serialised as JSON in the prompt; payloads in free text, fields, attachments, hidden text or tables
  layout_compat: *id001
  anchor: structured-query delimiting with an instruction hierarchy (D2)
evaluations:
- id: ev_o1_none
  defence_id: none:DeepSeek-V4-Flash, Qwen3.7-flash, Gemini 3.5 Flash-Lite and Mistral Small, pooled
  benchmark: 'municipal aid-procedure copilot, objective: eligibility manipulation (five vectors x five variants)'
  attacker_type: static
  attacker_access: black_box
  judge: deterministic match against ground truth and success marker, plus a validated gpt-5.6-terra judge
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 9
  decoding: temperature 0
  sample_size: '100'
  baseline_kind: instruction_prompt_only
- id: ev_o2_none
  defence_id: none:DeepSeek-V4-Flash, Qwen3.7-flash, Gemini 3.5 Flash-Lite and Mistral Small, pooled
  benchmark: 'municipal aid-procedure copilot, objective: requirement falsification (five vectors x five variants)'
  attacker_type: static
  attacker_access: black_box
  judge: deterministic match against ground truth and success marker, plus a validated gpt-5.6-terra judge
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 9
  decoding: temperature 0
  sample_size: '100'
  baseline_kind: instruction_prompt_only
- id: ev_o3_none
  defence_id: none:DeepSeek-V4-Flash, Qwen3.7-flash, Gemini 3.5 Flash-Lite and Mistral Small, pooled
  benchmark: 'municipal aid-procedure copilot, objective: summary falsification (five vectors x five variants)'
  attacker_type: static
  attacker_access: black_box
  judge: deterministic match against ground truth and success marker, plus a validated gpt-5.6-terra judge
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 9
  decoding: temperature 0
  sample_size: '100'
  baseline_kind: instruction_prompt_only
- id: ev_o4_none
  defence_id: none:DeepSeek-V4-Flash, Qwen3.7-flash, Gemini 3.5 Flash-Lite and Mistral Small, pooled
  benchmark: 'municipal aid-procedure copilot, objective: misrouting (five vectors x five variants)'
  attacker_type: static
  attacker_access: black_box
  judge: deterministic match against ground truth and success marker, plus a validated gpt-5.6-terra judge
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 9
  decoding: temperature 0
  sample_size: '100'
  baseline_kind: instruction_prompt_only
- id: ev_o5_none
  defence_id: none:DeepSeek-V4-Flash, Qwen3.7-flash, Gemini 3.5 Flash-Lite and Mistral Small, pooled
  benchmark: 'municipal aid-procedure copilot, objective: rule exfiltration (five vectors x five variants)'
  attacker_type: static
  attacker_access: black_box
  judge: deterministic match against ground truth and success marker, plus a validated gpt-5.6-terra judge
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 9
  decoding: temperature 0
  sample_size: '100'
  baseline_kind: instruction_prompt_only
- id: ev_o6_none
  defence_id: none:DeepSeek-V4-Flash, Qwen3.7-flash, Gemini 3.5 Flash-Lite and Mistral Small, pooled
  benchmark: 'municipal aid-procedure copilot, objective: fraudulent payment or contact insertion (five vectors x five variants)'
  attacker_type: static
  attacker_access: black_box
  judge: deterministic match against ground truth and success marker, plus a validated gpt-5.6-terra judge
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 9
  decoding: temperature 0
  sample_size: '100'
  baseline_kind: instruction_prompt_only
- id: ev_o1_dm
  defence_id: dm
  benchmark: 'municipal aid-procedure copilot, objective: eligibility manipulation (five vectors x five variants)'
  attacker_type: static
  attacker_access: black_box
  judge: deterministic match against ground truth and success marker, plus a validated gpt-5.6-terra judge
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 9
  decoding: temperature 0
  sample_size: '100'
- id: ev_o2_dm
  defence_id: dm
  benchmark: 'municipal aid-procedure copilot, objective: requirement falsification (five vectors x five variants)'
  attacker_type: static
  attacker_access: black_box
  judge: deterministic match against ground truth and success marker, plus a validated gpt-5.6-terra judge
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 9
  decoding: temperature 0
  sample_size: '100'
- id: ev_o3_dm
  defence_id: dm
  benchmark: 'municipal aid-procedure copilot, objective: summary falsification (five vectors x five variants)'
  attacker_type: static
  attacker_access: black_box
  judge: deterministic match against ground truth and success marker, plus a validated gpt-5.6-terra judge
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 9
  decoding: temperature 0
  sample_size: '100'
- id: ev_o4_dm
  defence_id: dm
  benchmark: 'municipal aid-procedure copilot, objective: misrouting (five vectors x five variants)'
  attacker_type: static
  attacker_access: black_box
  judge: deterministic match against ground truth and success marker, plus a validated gpt-5.6-terra judge
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 9
  decoding: temperature 0
  sample_size: '100'
- id: ev_o5_dm
  defence_id: dm
  benchmark: 'municipal aid-procedure copilot, objective: rule exfiltration (five vectors x five variants)'
  attacker_type: static
  attacker_access: black_box
  judge: deterministic match against ground truth and success marker, plus a validated gpt-5.6-terra judge
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 9
  decoding: temperature 0
  sample_size: '100'
- id: ev_o6_dm
  defence_id: dm
  benchmark: 'municipal aid-procedure copilot, objective: fraudulent payment or contact insertion (five vectors x five variants)'
  attacker_type: static
  attacker_access: black_box
  judge: deterministic match against ground truth and success marker, plus a validated gpt-5.6-terra judge
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 9
  decoding: temperature 0
  sample_size: '100'
- id: ev_o1_sq
  defence_id: sq
  benchmark: 'municipal aid-procedure copilot, objective: eligibility manipulation (five vectors x five variants)'
  attacker_type: static
  attacker_access: black_box
  judge: deterministic match against ground truth and success marker, plus a validated gpt-5.6-terra judge
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 9
  decoding: temperature 0
  sample_size: '100'
- id: ev_o2_sq
  defence_id: sq
  benchmark: 'municipal aid-procedure copilot, objective: requirement falsification (five vectors x five variants)'
  attacker_type: static
  attacker_access: black_box
  judge: deterministic match against ground truth and success marker, plus a validated gpt-5.6-terra judge
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 9
  decoding: temperature 0
  sample_size: '100'
- id: ev_o3_sq
  defence_id: sq
  benchmark: 'municipal aid-procedure copilot, objective: summary falsification (five vectors x five variants)'
  attacker_type: static
  attacker_access: black_box
  judge: deterministic match against ground truth and success marker, plus a validated gpt-5.6-terra judge
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 9
  decoding: temperature 0
  sample_size: '100'
- id: ev_o4_sq
  defence_id: sq
  benchmark: 'municipal aid-procedure copilot, objective: misrouting (five vectors x five variants)'
  attacker_type: static
  attacker_access: black_box
  judge: deterministic match against ground truth and success marker, plus a validated gpt-5.6-terra judge
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 9
  decoding: temperature 0
  sample_size: '100'
- id: ev_o5_sq
  defence_id: sq
  benchmark: 'municipal aid-procedure copilot, objective: rule exfiltration (five vectors x five variants)'
  attacker_type: static
  attacker_access: black_box
  judge: deterministic match against ground truth and success marker, plus a validated gpt-5.6-terra judge
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 9
  decoding: temperature 0
  sample_size: '100'
- id: ev_o6_sq
  defence_id: sq
  benchmark: 'municipal aid-procedure copilot, objective: fraudulent payment or contact insertion (five vectors x five variants)'
  attacker_type: static
  attacker_access: black_box
  judge: deterministic match against ground truth and success marker, plus a validated gpt-5.6-terra judge
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 9
  decoding: temperature 0
  sample_size: '100'
- id: ev_all_none
  defence_id: none:DeepSeek-V4-Flash, Qwen3.7-flash, Gemini 3.5 Flash-Lite and Mistral Small, pooled
  benchmark: municipal aid-procedure copilot, all six objectives
  attacker_type: static
  attacker_access: black_box
  judge: deterministic match against ground truth and success marker, plus a validated gpt-5.6-terra judge
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 7
  decoding: temperature 0
  sample_size: '600'
  baseline_kind: instruction_prompt_only
- id: ev_fid_none
  defence_id: none:DeepSeek-V4-Flash, Qwen3.7-flash, Gemini 3.5 Flash-Lite and Mistral Small, pooled
  benchmark: legitimate, unpoisoned case files
  attacker_type: none
  attacker_access: not_applicable
  judge: all authoritative output fields match the rule-based ground truth
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 6
  decoding: temperature 0
  sample_size: '200'
  baseline_kind: instruction_prompt_only
- id: ev_all_dm
  defence_id: dm
  benchmark: municipal aid-procedure copilot, all six objectives
  attacker_type: static
  attacker_access: black_box
  judge: deterministic match against ground truth and success marker, plus a validated gpt-5.6-terra judge
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 7
  decoding: temperature 0
  sample_size: '600'
- id: ev_fid_dm
  defence_id: dm
  benchmark: legitimate, unpoisoned case files
  attacker_type: none
  attacker_access: not_applicable
  judge: all authoritative output fields match the rule-based ground truth
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 6
  decoding: temperature 0
  sample_size: '200'
- id: ev_all_sq
  defence_id: sq
  benchmark: municipal aid-procedure copilot, all six objectives
  attacker_type: static
  attacker_access: black_box
  judge: deterministic match against ground truth and success marker, plus a validated gpt-5.6-terra judge
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 7
  decoding: temperature 0
  sample_size: '600'
- id: ev_fid_sq
  defence_id: sq
  benchmark: legitimate, unpoisoned case files
  attacker_type: none
  attacker_access: not_applicable
  judge: all authoritative output fields match the rule-based ground truth
  layout_same_as_defence: 'yes'
  measured_by: municipal_sauc2026
  page: 6
  decoding: temperature 0
  sample_size: '200'
results:
- id: r_o1_none
  evaluation_id: ev_o1_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, eligibility manipulation
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_o2_none
  evaluation_id: ev_o2_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, requirement falsification
  value: '6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_o3_none
  evaluation_id: ev_o3_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, summary falsification
  value: '8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_o4_none
  evaluation_id: ev_o4_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, misrouting
  value: '38'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_o5_none
  evaluation_id: ev_o5_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, rule exfiltration
  value: '17'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_o6_none
  evaluation_id: ev_o6_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fraudulent payment or contact insertion
  value: '26'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_o1_dm
  evaluation_id: ev_o1_dm
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, eligibility manipulation
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_o2_dm
  evaluation_id: ev_o2_dm
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, requirement falsification
  value: '7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_o3_dm
  evaluation_id: ev_o3_dm
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, summary falsification
  value: '3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_o4_dm
  evaluation_id: ev_o4_dm
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, misrouting
  value: '33'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_o5_dm
  evaluation_id: ev_o5_dm
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, rule exfiltration
  value: '13'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_o6_dm
  evaluation_id: ev_o6_dm
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fraudulent payment or contact insertion
  value: '13'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_o1_sq
  evaluation_id: ev_o1_sq
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, eligibility manipulation
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_o2_sq
  evaluation_id: ev_o2_sq
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, requirement falsification
  value: '6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_o3_sq
  evaluation_id: ev_o3_sq
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, summary falsification
  value: '6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_o4_sq
  evaluation_id: ev_o4_sq
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, misrouting
  value: '26'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_o5_sq
  evaluation_id: ev_o5_sq
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, rule exfiltration
  value: '15'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_o6_sq
  evaluation_id: ev_o6_sq
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, fraudulent payment or contact insertion
  value: '12'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 9
  location: Table 4
- id: r_all_none
  evaluation_id: ev_all_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, all objectives
  value: '15.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_fid_none
  evaluation_id: ev_fid_none
  quantity: R2clean
  coverage: direct
  metric_as_printed: fidelity on legitimate cases
  value: '80.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_all_dm
  evaluation_id: ev_all_dm
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, all objectives
  value: '11.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_fid_dm
  evaluation_id: ev_fid_dm
  quantity: R2clean
  coverage: direct
  metric_as_printed: fidelity on legitimate cases
  value: '59.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
- id: r_all_sq
  evaluation_id: ev_all_sq
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, all objectives
  value: '10.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 7
  location: Table 3
- id: r_fid_sq
  evaluation_id: ev_fid_sq
  quantity: R2clean
  coverage: direct
  metric_as_printed: fidelity on legitimate cases
  value: '78.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table 2
comparisons:
- id: c_o1_dm
  left: ev_o1_dm
  right: ev_o1_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 9
- id: c_o2_dm
  left: ev_o2_dm
  right: ev_o2_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 9
- id: c_o3_dm
  left: ev_o3_dm
  right: ev_o3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 9
- id: c_o4_dm
  left: ev_o4_dm
  right: ev_o4_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 9
- id: c_o5_dm
  left: ev_o5_dm
  right: ev_o5_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 9
- id: c_o6_dm
  left: ev_o6_dm
  right: ev_o6_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 9
- id: c_all_dm
  left: ev_all_dm
  right: ev_all_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_fid_dm
  left: ev_fid_dm
  right: ev_fid_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_o1_sq
  left: ev_o1_sq
  right: ev_o1_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 9
- id: c_o2_sq
  left: ev_o2_sq
  right: ev_o2_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 9
- id: c_o3_sq
  left: ev_o3_sq
  right: ev_o3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 9
- id: c_o4_sq
  left: ev_o4_sq
  right: ev_o4_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 9
- id: c_o5_sq
  left: ev_o5_sq
  right: ev_o5_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 9
- id: c_o6_sq
  left: ev_o6_sq
  right: ev_o6_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 9
- id: c_all_sq
  left: ev_all_sq
  right: ev_all_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_fid_sq
  left: ev_fid_sq
  right: ev_fid_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
claims:
- id: k1
  text: defences reduce attack success unevenly across objectives, barely affecting summary falsification
  location: abstract
  anchor: Defences reduce attack success, but unevenly across objectives
  jobs:
  - R1
  supported_by:
  - c_o1_dm
  - c_o2_dm
  - c_o3_dm
  - c_o4_dm
  - c_o5_dm
  - c_o6_dm
  - c_o1_sq
  - c_o2_sq
  - c_o3_sq
  - c_o4_sq
  - c_o5_sq
  - c_o6_sq
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 3 and 4
    R2clean: Table 2
    R2attack: not reported
schema_gaps:
- where: report
  note: results are pooled over four models with 100 attacks per objective and defence cell; datamarking lowers fidelity on legitimate cases from 80.5% to 59.0%

## Stage 2. The PDFs read are the PDFs registered, and every report is an included study

In [ ]:
import csv, logging, datetime as dt
from collections import Counter
import schema, search, corpus, paths
for m in (schema, search, corpus, paths):
    importlib.reload(m)
logging.getLogger("pypdf").setLevel(logging.ERROR)
from pypdf import PdfReader

BATCH = "04k"
cfg_exclusions = 2
cfg_next = "notebook 04 (full extraction: assembly of the five tables, second pass and agreement)"
IDS = ['haaulc_rs2026', 'peerrevcheck_osf2025', 'spotguard_applsci2026', 'advershield_jtsi2026', 'hybridrag_ijaiml2026', 'eroded_zenodo2026', 'municipal_sauc2026']
man = {m["slug"]: m for m in search.read_csv(paths.REPORTS / "defences.csv")}
problems, texts = [], {}
for rid in IDS:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    pdf = paths.DATA_DEFENCES / f"{slug}.pdf"
    m = man.get(slug, {})
    if not pdf.exists():
        problems.append(f"{rid}: {pdf.name} not in data/defences"); continue
    sha = corpus.sha256(pdf)
    rd = PdfReader(str(pdf))
    texts[slug] = ([p.extract_text() or "" for p in rd.pages], [p.extract_text(extraction_mode="layout") or "" for p in rd.pages])
    ok_sha = sha == m.get("sha256")
    ok_inc = m.get("decision") == "include"
    print(f"{rid:<28} {slug:<12} pages {len(rd.pages):>3}  sha256 {'matches' if ok_sha else 'DIFFERS'}  decision {m.get('decision')}")
    if not ok_sha:
        problems.append(f"{rid}: PDF differs from the one registered in reports/defences.csv")
    if not ok_inc:
        problems.append(f"{rid}: not an included study")

## Stage 2b. Full-text exclusions found while coding

Each study below was included at title and abstract; its full text shows it does not meet the inclusion rule. The anchor is checked in its PDF, then the decision is written to schema/fulltext_decisions.csv and schema/screening_decisions.csv and carried into reports/screening.csv, reports/defences.csv, reports/fulltext_check.csv and reports/prisma_counts.json, as notebook 02d did.

In [ ]:
import json
import overlap
importlib.reload(overlap)
EXCL = [{'key': 'doi:10.21203/rs.3.rs-11711485/v1', 'slug': 'doi_10_21203_rs_3_rs_11711485_v1', 'reason': 'system level', 'anchor': 'This asymmetry is load-bearing', 'note': 'Signed to Stop: the defence is a pinned operator key with an operator-side signature-verification tool, and the agent honours only halts the tool verifies; its protection rests on a trusted tool outside the model, as the authors state'}, {'key': 'doi:10.3126/joeis.v4i1.81604', 'slug': 'doi_10_3126_joeis_v4i1_81604', 'reason': 'system level', 'anchor': 'A preprocessing layer scanned and filtered inputs for potentially malicious content', 'note': 'the only defended arm combines adversarial retraining with an input-sanitisation filter, and direct and indirect attacks are pooled'}]
FT_FIELDS = ["key", "slug", "decision", "reason", "basis", "anchor", "note"]
SD_FIELDS = ["key", "decision", "reason", "stage", "access_status", "note", "title"]
CHECK_FIELDS = ["key", "slug", "decision", "reason", "basis", "page", "status", "anchor", "sha256", "note"]
ft = search.read_csv(paths.SCHEMA / "fulltext_decisions.csv")
sd = search.read_csv(paths.SCHEMA / "screening_decisions.csv")
fchk = search.read_csv(paths.REPORTS / "fulltext_check.csv")
mlist = search.read_csv(paths.REPORTS / "defences.csv")
for e in EXCL:
    m = next((x for x in mlist if x["slug"] == e["slug"]), None)
    pdf = paths.DATA_DEFENCES / f"{e['slug']}.pdf"
    if m is None or not pdf.exists():
        problems.append(f"exclusion {e['slug']}: not in reports/defences.csv or PDF missing"); continue
    sha = corpus.sha256(pdf)
    if sha != m.get("sha256"):
        problems.append(f"exclusion {e['slug']}: PDF differs from the one registered")
    if e["reason"] not in search.EXCLUSION_REASONS:
        problems.append(f"exclusion {e['slug']}: reason not in the fixed list")
    pages = overlap.pdf_page_texts(pdf)
    a = overlap.norm_text(e["anchor"])
    hits = [i + 1 for i, t in enumerate(pages) if a in t]
    status = "found" if hits else "anchor not found"
    if not hits:
        problems.append(f"exclusion {e['slug']}: anchor not found in the PDF")
    row = {"key": e["key"], "slug": e["slug"], "decision": "exclude", "reason": e["reason"], "basis": "full text", "anchor": e["anchor"], "note": e["note"]}
    ft = [r for r in ft if r["key"] != e["key"]] + [row]
    fchk = [r for r in fchk if r["key"] != e["key"]] + [dict(row, page=str(hits[0]) if hits else "", status=status, sha256=sha)]
    n = 0
    for r in sd:
        if r["key"] == e["key"]:
            r.update(decision="exclude", reason=e["reason"], stage="full_text", note="full text: " + e["note"]); n += 1
    if n != 1:
        problems.append(f"exclusion {e['slug']}: {n} rows for {e['key']} in schema/screening_decisions.csv")
    m.update(decision="exclude", reason=e["reason"], stage="full_text", note="full text: " + e["note"])
    print(f"{e['slug']:<12} excluded at full text ({e['reason']}); anchor {status}{' on page ' + str(hits[0]) if hits else ''}")
def write_lf(path, rows, fields):              # the schema files were written with plain newlines; keep them that way
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore", lineterminator="\n")
        w.writeheader()
        w.writerows({k: r.get(k) or "" for k in fields} for r in rows)
write_lf(paths.SCHEMA / "fulltext_decisions.csv", ft, FT_FIELDS)
write_lf(paths.SCHEMA / "screening_decisions.csv", sd, SD_FIELDS)
search.write_csv(paths.REPORTS / "fulltext_check.csv", fchk, CHECK_FIELDS)
search.write_csv(paths.REPORTS / "defences.csv", mlist, list(mlist[0].keys()))
rows = search.read_csv(paths.REPORTS / "screening.csv")
rows, more = search.apply_decisions(rows, sd)
problems += more
search.write_csv(paths.REPORTS / "screening.csv", rows, search.SCREEN_FIELDS)
prisma = json.loads((paths.REPORTS / "prisma_counts.json").read_text())
prisma.update(search.prisma_counts(search.read_csv(paths.REPORTS / "search_log.csv"), rows))
prisma["fulltext_anchors_found"] = sum(1 for r in fchk if r["status"].startswith("found"))
(paths.REPORTS / "prisma_counts.json").write_text(json.dumps(prisma, indent=1))
print(f"included now: {prisma['included']}; full text assessed: {prisma['full_text_assessed']}; "
      f"full-text exclusions: {json.dumps(prisma['excluded_full_text_by_reason'])}")

## Stage 3. Validate against the frozen schema and the PDFs; grade the comparisons; extra checks

In [ ]:
import re
import pypdfium2 as pdfium
_docs = {}
def pdfium_page(slug, page):
    if slug not in _docs:
        _docs[slug] = pdfium.PdfDocument(str(paths.DATA_DEFENCES / f"{slug}.pdf"))
    return _docs[slug][page - 1].get_textpage().get_text_range()
S1 = schema.load_schema(paths.SCHEMA / "record_schema.yaml")
check_rows, summary, residual, third_rows = [], [], [], []
for rid in IDS:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    plain, lay = texts.get(slug, (None, None))
    probs, notes = schema.validate_record(rec, S1, plain, lay)
    kept, third = [], []                       # a value pypdf glues to its neighbour is looked for in a third extraction (pypdfium2)
    for p in probs:
        m = re.match(r"results:(\S+): value '(.+)' not found on page (\d+)$", p)
        if m and schema.value_on_page(m.group(2), pdfium_page(slug, int(m.group(3)))):
            third.append(f"{m.group(1)} ({m.group(2)!r}, page {m.group(3)})")
        else:
            kept.append(p)
    probs = kept
    for kind in ("defences", "evaluations", "results", "comparisons", "claims"):          # ids unique within each kind
        dup = [i for i, n in Counter(r["id"] for r in rec.get(kind) or []).items() if n > 1]
        probs += [f"{kind}: id {i} used more than once" for i in dup]
    fig_kept, third_chart = 0, []
    for r in rec.get("results") or []:                                                     # chart values kept as printed
        if r.get("value_source") == "figure_only" and r.get("value") not in (None, ""):
            fig_kept += 1
            pg = int(r["page"]) - 1
            if plain and not (schema.value_on_page(r["value"], plain[pg]) or schema.value_on_page(r["value"], lay[pg])):
                if schema.value_on_page(r["value"], pdfium_page(slug, int(r["page"]))):
                    third_chart.append(f"{r['id']} ({r['value']!r}, page {r['page']}, chart label)")
                else:
                    probs.append(f"results:{r['id']}: chart value {r['value']!r} not found on page {r['page']}")
    third_rows += [{"report": rid, "status": "value found in the third extraction", "detail": t} for t in third + third_chart]
    grades = Counter(schema.grade(c) for c in rec.get("comparisons") or [])
    n_val = sum(1 for r in rec.get("results") or [] if r.get("value_source") != "figure_only")
    n_anc = sum(1 for k in ("defences", "claims") for r in rec.get(k) or [] if r.get("anchor"))
    summary.append({"batch": BATCH, "report": rid, "slug": slug, "role": rec["report"]["role"],
                    "defences": len(rec.get("defences") or []), "evaluations": len(rec.get("evaluations") or []),
                    "results": len(rec.get("results") or []), "values_checked": n_val, "chart_values_checked": fig_kept,
                    "anchors_checked": n_anc, "comparisons": len(rec.get("comparisons") or []),
                    "grade_A": grades["A"], "grade_B": grades["B"], "grade_C": grades["C"], "grade_U": grades["U"],
                    "claims": len(rec.get("claims") or []),
                    **{f"cov_{q}": rec["coverage"][q] for q in ("R1", "R2clean", "R2attack", "R3ns", "R3cp", "R3safe")},
                    "values_third_extraction": len(third), "chart_values_third_extraction": len(third_chart), "residual_notes": len(notes), "problems": len(probs)})
    check_rows += [{"report": rid, "status": "problem", "detail": p} for p in probs]
    residual += [{"report": rid, "where": n.split(": ", 1)[0], "note": n.split(": ", 1)[1]} for n in notes]
    print(f"{rid:<28} problems {len(probs)}; values {n_val} ({len(third)} via the third extraction); chart values {fig_kept} ({len(third_chart)} via the third extraction); anchors {n_anc}; grades {dict(grades)}")
    for p in probs:
        print("   ", p)
    problems += [f"{rid}: {p}" for p in probs]
check_rows += third_rows + [{"report": g["report"], "status": "residual note", "detail": f"{g['where']}: {g['note']}"} for g in residual]
search.write_csv(paths.REPORTS / f"extraction_{BATCH}_check.csv", check_rows or [{"report": "all", "status": "ok", "detail": "no problems"}],
                 ["report", "status", "detail"])
search.write_csv(paths.REPORTS / f"extraction_{BATCH}_summary.csv", summary, list(summary[0].keys()))
for g in residual:
    print(f"  residual note, {g['report']}: {g['note'][:150]}")

## Stage 4. Batch register (one row per coded report, kept across batches)

In [ ]:
reg_path = paths.REPORTS / "extraction_batches.csv"
reg = [r for r in search.read_csv(reg_path) if r["batch"] != BATCH] if reg_path.exists() else []
today = dt.date.today().isoformat()
for s in summary:
    reg.append({"batch": BATCH, "report": s["report"], "slug": s["slug"], "coded": today,
                "status": "checked" if s["problems"] == 0 else "problems"})
reg.sort(key=lambda r: (r["batch"], r["report"]))
search.write_csv(reg_path, reg, ["batch", "report", "slug", "coded", "status"])
included = sum(1 for m in search.read_csv(paths.REPORTS / "defences.csv") if m.get("decision") == "include")   # read again: Stage 2b may change it
pilot = len(list(paths.RECORDS.glob("*.yaml"))) - len({r["report"] for r in reg})
print(f"records in reports/records: {len(list(paths.RECORDS.glob('*.yaml')))} (pilot {pilot}, batches {len({r['report'] for r in reg})}); included studies: {included}")

## Stage 5. Decision record and tests

In [ ]:
tot = {k: sum(s[k] for s in summary) for k in ("defences", "evaluations", "results", "values_checked", "values_third_extraction", "chart_values_checked",
                                              "chart_values_third_extraction", "anchors_checked", "comparisons", "grade_A", "grade_B", "grade_C", "grade_U", "claims")}
status = "BATCH CHECKED" if not problems else "BATCH NOT CHECKED: PROBLEMS BELOW"
lines = [f"decision_{BATCH}: {status}", f"this step: {dt.date.today().isoformat()}",
         "reports: " + ", ".join(IDS),
         f"records: {tot['defences']} defences, {tot['evaluations']} evaluations, {tot['results']} results, "
         f"{tot['comparisons']} comparisons, {tot['claims']} claims",
         f"checked against the PDFs: {tot['values_checked']} printed values ({tot['values_third_extraction']} of them found only in the third "
         f"extraction), {tot['chart_values_checked']} chart values ({tot['chart_values_third_extraction']} of them found only in the third extraction) "
         f"and {tot['anchors_checked']} anchors, each found on its page",
         f"comparison grades: A {tot['grade_A']}, B {tot['grade_B']}, C {tot['grade_C']}, U {tot['grade_U']}",
         f"residual notes: {len(residual)}; full-text exclusions found while coding: {cfg_exclusions}",
         f"next: {cfg_next}"] + [f"- problem: {p}" for p in problems]
(paths.REPORTS / f"decision_{BATCH}.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout.strip().splitlines()[-1] if tests.stdout.strip() else tests.stderr.strip()[-400:])

## Stage 6. Commit

In [ ]:
import os, subprocess
os.chdir(os.environ.get("SOURCEMAP_REPO_ROOT", "/content/drive/MyDrive/SourceMap_Research/sourcemap-research"))
r = subprocess.run(["python", "tools/commit_cell.py",
                    "04k extraction batch k: seven included studies coded from their PDFs and two full-text exclusions, values and anchors checked, comparisons graded"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)